In [ ]:
import cv2
import os
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
recognizer = cv2.face.LBPHFaceRecognizer_create(
    radius=2, 
    neighbors=16, 
    grid_x=8, 
    grid_y=8
)

def get_data(path):
    image_paths = [os.path.join(path, f) for f in os.listdir(path)]
    faces = []
    ids = []
    
    # CLAHE helps to improve contrast locally (better than global equalization)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))

    for img_path in image_paths:
        try:
            # 1. Load image and convert to grayscale
            img = Image.open(img_path).convert('L') 
            img_np = np.array(img, 'uint8')
            
            # 2. Extract ID
            user_id = int(os.path.split(img_path)[-1].split(".")[1])
            
            # 3. Pre-processing: Smoothing + Contrast Enhancement
            # Bilateral filter removes noise but keeps edges (eyes/nose/mouth) sharp
            smoothed = cv2.bilateralFilter(img_np, 5, 75, 75)
            enhanced = clahe.apply(smoothed)
            
            # 4. Augmentation: Add the original processed image
            faces.append(enhanced)
            ids.append(user_id)
            
            # 5. Augmentation: Add a horizontally flipped version
            # This helps if the person is facing a slightly different way
            flipped_img = cv2.flip(enhanced, 1)
            faces.append(flipped_img)
            ids.append(user_id)
            
        except Exception as e:
            print(f"Skipping {img_path}: {e}")
            
    return faces, ids

# Load and Split
data_path = r'C:\Users\hp\Desktop\Attendance-System-Using-Face-Recognition\Dataset\training\Cleaned_Training'
all_faces, all_ids = get_data(data_path)

X_train, X_test, y_train, y_test = train_test_split(
    all_faces, all_ids, test_size=0.3, random_state=43
)

# Train
recognizer.train(X_train, np.array(y_train))
print(f"Model trained on {len(X_train)} images.")

# Evaluate
correct = 0
total = len(X_test)

for i in range(total):
    predicted_id, confidence = recognizer.predict(X_test[i])
    
    # IMPROVEMENT: Adding a confidence threshold
    # If confidence > 150, it's likely a bad match regardless of the ID
    if predicted_id == y_test[i] :
        correct += 1
        result = "MATCH"
    else:
        result = "WRONG"
        
    print(f"Actual: {y_test[i]} | Predicted: {predicted_id} | Conf: {confidence:.2f} | {result}")

accuracy = (correct / total) * 100
print(f"\nFinal Accuracy: {accuracy:.2f}%")

Model trained on 190 images.
Actual: 2 | Predicted: 2 | Conf: 93.50 | MATCH
Actual: 4 | Predicted: 4 | Conf: 105.35 | MATCH
Actual: 6 | Predicted: 6 | Conf: 85.58 | MATCH
Actual: 1 | Predicted: 1 | Conf: 108.70 | MATCH
Actual: 5 | Predicted: 5 | Conf: 101.94 | MATCH
Actual: 2 | Predicted: 2 | Conf: 102.68 | MATCH
Actual: 6 | Predicted: 6 | Conf: 99.58 | MATCH
Actual: 4 | Predicted: 4 | Conf: 108.00 | MATCH
Actual: 2 | Predicted: 2 | Conf: 108.21 | MATCH
Actual: 3 | Predicted: 3 | Conf: 92.10 | MATCH
Actual: 1 | Predicted: 1 | Conf: 100.19 | MATCH
Actual: 4 | Predicted: 3 | Conf: 113.76 | WRONG
Actual: 4 | Predicted: 4 | Conf: 122.99 | MATCH
Actual: 3 | Predicted: 6 | Conf: 120.64 | WRONG
Actual: 3 | Predicted: 3 | Conf: 96.18 | MATCH
Actual: 2 | Predicted: 2 | Conf: 75.09 | MATCH
Actual: 4 | Predicted: 3 | Conf: 112.77 | WRONG
Actual: 1 | Predicted: 6 | Conf: 107.70 | WRONG
Actual: 4 | Predicted: 3 | Conf: 116.43 | WRONG
Actual: 2 | Predicted: 2 | Conf: 103.24 | MATCH
Actual: 2 | Predi